# System Looting — free public server (Google Colab + playit.gg)

Runs the **System Looting** Luanti server on Google's free Colab machines and
exposes it to the internet through a **playit.gg** tunnel, so anyone can join
from a normal Luanti client. **No credit card needed** — just a Google account.

**Limits (free tier):** sessions run up to ~12 h, disconnect after ~90 min of
inactivity, and the VM is wiped afterwards. Great for play sessions, not for a
24/7 server (see `docs/FREE_HOSTING.md` for Oracle free tier etc.).

**How:** Runtime → *Run all*. Then follow the instructions printed at the end
(one-time playit account claim).

In [ ]:
# 1) Engine: use an existing install, else apt (Ubuntu repo, then the PPA as fallback)
import subprocess, shutil, os

def find_engine():
    # shutil.which does NOT search /usr/games (where Debian/Ubuntu put the
    # minetest/luanti server), so check it explicitly.
    for c in ('luantiserver', 'minetestserver', 'minetest'):
        p = shutil.which(c)
        if p:
            return p
    for p in ('/usr/games/luantiserver', '/usr/games/minetestserver',
              '/usr/bin/luantiserver', '/usr/bin/minetestserver'):
        if os.path.exists(p):
            return p
    return None

engine = find_engine()
if not engine:
    print('no engine found — installing minetest-server …')
    r = subprocess.run('apt-get update -qq && apt-get install -y -qq minetest-server',
                       shell=True, capture_output=True, text=True, timeout=900)
    if r.returncode != 0:
        print('Ubuntu repo failed — trying the minetestdevs PPA …')
        subprocess.run('apt-get install -y -qq software-properties-common', shell=True, timeout=300)
        subprocess.run('add-apt-repository -y ppa:minetestdevs/stable', shell=True, timeout=600)
        subprocess.run('apt-get update -qq && apt-get install -y -qq minetest-server',
                       shell=True, timeout=900)
    engine = find_engine()
assert engine, 'engine install failed — check the messages above'
print('engine:', engine)
print(subprocess.run(f'{engine} --version', shell=True, capture_output=True,
                     text=True).stdout.splitlines()[0])

In [ ]:
# 2) Game + world + start the server on UDP 30000
import subprocess, shutil, time, os

def find_engine():
    for c in ('luantiserver', 'minetestserver', 'minetest'):
        p = shutil.which(c)
        if p:
            return p
    for p in ('/usr/games/luantiserver', '/usr/games/minetestserver',
              '/usr/bin/luantiserver', '/usr/bin/minetestserver'):
        if os.path.exists(p):
            return p
    return None

GAME = '/content/SystemTest'   # folder name MUST equal the game id
os.makedirs('/content', exist_ok=True)
if not os.path.isdir(f'{GAME}/.git'):
    subprocess.run('git clone --depth 1 https://github.com/SodoMita/SystemTest '
                   f'{GAME}', shell=True, check=True, timeout=300)
else:
    subprocess.run(f'git -C {GAME} pull --ff-only', shell=True, timeout=120)

# ALSO expose it under ~/.minetest/games and ~/.luanti/games (the engine only
# searches those dirs; it derives the game id from the FOLDER NAME).
for base in (os.path.expanduser('~/.minetest/games'), os.path.expanduser('~/.luanti/games')):
    os.makedirs(base, exist_ok=True)
    link = os.path.join(base, 'SystemTest')
    if os.path.islink(link):
        os.remove(link)
    elif os.path.exists(link):
        os.system(f'rm -rf {link}')
    os.symlink(GAME, link)
    print('linked:', link, '->', GAME)

os.makedirs(f'{GAME}/worlds/systemloot', exist_ok=True)
open(f'{GAME}/worlds/systemloot/world.mt', 'w').write(
    'gameid = SystemTest\nbackend = sqlite3\nmg_name = singlenode\n')
open(f'{GAME}/systemloot.conf', 'w').write('''
server_name = System Looting — Colab
server_description = Free Colab test server
port = 30000
bind_address = 0.0.0.0
max_users = 16
mg_name = singlenode
time_speed = 0
enable_damage = true
sl_auto_start = true
sl_auto_start_delay = 20
''')

engine = find_engine()
assert engine, 'no engine — run cell 1 first'
print('engine:', engine)

open(f'{GAME}/server.log', 'w').close()
open(f'{GAME}/server.err', 'w').close()
proc = subprocess.Popen(
    f'{engine} --gameid SystemTest --world {GAME}/worlds/systemloot '
    f'--config {GAME}/systemloot.conf --logfile {GAME}/server.log --port 30000',
    shell=True, stdout=subprocess.DEVNULL, stderr=open(f'{GAME}/server.err', 'a'))

log = ''
for _ in range(25):
    time.sleep(1)
    try:
        log = open(f'{GAME}/server.log').read()
    except FileNotFoundError:
        continue
    if 'listening on' in log:
        break

if 'listening on' in log:
    print('SERVER UP ✔  port 30000')
    print('\n'.join(l for l in log.splitlines()
                  if 'listening' in l or 'Loaded core' in l or 'arena' in l))
else:
    print('SERVER DID NOT START — process alive:', proc.poll() is None)
    print('--- server.log ---')
    print(log[-2000:] if log else '(empty)')
    print('--- server.err ---')
    print(open(f'{GAME}/server.err').read()[-2000:] or '(empty)')

In [ ]:
# 3) playit.gg tunnel → public address
import subprocess, time, os, re
if not os.path.exists('/content/playit'):
    subprocess.run('curl -SsL https://playit.gg/download/playit-linux-amd64 -o /content/playit && chmod +x /content/playit',
                   shell=True, check=True, timeout=300)
subprocess.Popen('script -q -c /content/playit /content/playit.log', shell=True,
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(12)
log = open('/content/playit.log', errors='replace').read()
claim = re.search(r'https://playit\.gg/claim/[A-Za-z0-9]+', log)
print('=' * 66)
print('  PUBLIC SERVER — NEXT STEPS (once, ~2 minutes):')
print('=' * 66)
if claim:
    print('  1. OPEN THIS CLAIM LINK in any browser:')
    print('     ' + claim.group(0))
    print('     (create the free playit account — no card needed)')
else:
    print('  1. Run:  cat /content/playit.log   and look for the claim URL.')
print('  2. After claiming: playit.gg dashboard → add tunnel:')
print('     Type UDP, local port 30000  (and one TCP 30000)')
print('  3. Share the public address playit shows (e.g. 123.45.67.89:51234).')
print('  4. Server log any time:  tail /content/SystemTest/server.log')
print('=' * 66)

### After claiming

1. playit.gg dashboard → add tunnel: **UDP**, local port **30000** (and one **TCP** 30000).
2. playit shows a public address like `123.45.67.89:51234` — share it; anyone joins from Luanti.
3. Keep this tab open; interact occasionally (idle >90 min disconnects; sessions end ~12 h — rerun *Run all*; world resets unless backed up, see next cell).

For a permanent free server: `docs/FREE_HOSTING.md` (Oracle Always Free).

In [ ]:
# Optional: back up the world to Google Drive
# from google.colab import drive; drive.mount('/content/drive')
# import subprocess, os
# os.makedirs('/content/drive/MyDrive/systemloot', exist_ok=True)
# subprocess.run('cp -r /content/game/worlds/systemloot /content/drive/MyDrive/systemloot/', shell=True)
# print('world backed up')